# Prompting Techniques with Ollama

Prompting steers a pretrained LLM at inference time, with **no weight updates**. In this notebook you will:

1. See four techniques in action: **few-shot**, **chain-of-thought**, **self-consistency**, **RAG**
2. Practice them yourself in short exercises

| Technique | Idea |
|---|---|
| Zero/Few-shot | Give instructions, optionally with examples |
| Chain-of-Thought (CoT) | Ask for step-by-step reasoning before the answer |
| Self-Consistency | Sample several CoT chains, majority-vote the answer |
| RAG | Retrieve relevant documents and put them in the prompt |

## Setup
Run once (Colab with GPU runtime). It installs Ollama and pulls a small model.

In [1]:
%pip install -q ollama scikit-learn numpy
!sudo apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!nohup ollama serve > ollama.log 2>&1 &
import time; time.sleep(5)
!ollama pull gemma4:e2b

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 6.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 



In [2]:
import ollama, re
from collections import Counter

MODEL = "gemma4:e2b"

def generate(prompt, max_tokens=150, temperature=0.0):
    """Send a prompt to the model and return its text. temperature=0 is deterministic."""
    r = ollama.generate(model=MODEL, prompt=prompt, think=False,
                        options={"temperature": temperature, "num_predict": max_tokens})
    return r["response"].strip()

print(generate("Say hello in one short sentence."))

Hello!


## Demo 1: Zero-shot vs. Few-shot
Few-shot adds input→output examples so the model copies the format.

In [3]:
review = "The pacing was dreadful, but the soundtrack was gorgeous."
task = "Classify the review as Positive, Negative, or Mixed. Reply with one word."

zero_shot = f"{task}\n\nReview: {review}\nLabel:"

few_shot = f"""{task}

Review: A masterpiece of modern cinema.
Label: Positive

Review: I fell asleep halfway through.
Label: Negative

Review: Great acting, but the plot made no sense.
Label: Mixed

Review: {review}
Label:"""

print("Zero-shot:", generate(zero_shot, max_tokens=10))
print("Few-shot :", generate(few_shot, max_tokens=10))

Zero-shot: Mixed
Few-shot : Mixed


## Demo 2: Chain-of-Thought
Asking the model to reason step by step helps on multi-step problems.

In [4]:
question = ("A store had 50 shirts. It sold 18 on Monday, got a shipment of 30, "
            "then sold half of what it had. How many shirts are left?")   # answer: 31

print("Direct:\n", generate(f"Q: {question}\nGive only the final number.\nA:", max_tokens=10))
print("\nZero-shot CoT:\n", generate(f"Q: {question}\nA: Let's think step by step.", max_tokens=200))

Direct:
 36

Zero-shot CoT:
 Here is the step-by-step solution:

**1. Starting Inventory:**
The store begins with **50** shirts.

**2. Sales on Monday:**
They sold 18 shirts.
$50 - 18 = 32$ shirts remaining.

**3. Shipment Received:**
They received a shipment of 30 shirts.
$32 + 30 = 62$ shirts total.

**4. Second Sale:**
They sold half of what they had (half of 62).
Half of 62 is $62 / 2 = 31$ shirts sold.

**5. Final Count:**
Subtract the second sale from the current total:
$62 - 31 = 31$ shirts left.

**Answer:** There are **31** shirts left.


## Demo 3: Self-Consistency
Sample several reasoning chains (temperature > 0) and take the most common final answer.

In [5]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

prompt = (f"Q: {question}\nThink step by step, then end with 'The answer is <number>.'\nA:")
answers = []
for _ in range(5):
    out = generate(prompt, max_tokens=250, temperature=0.8)
    answers.append(final_number(out))

print("Votes :", Counter(answers))
print("Winner:", Counter(answers).most_common(1)[0][0], "(truth: 31)")

Votes : Counter({'31': 5})
Winner: 31 (truth: 31)


## Demo 4: RAG
Retrieve the most relevant document (TF-IDF here; real systems use embeddings + a vector DB) and add it to the prompt.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "Acme Corp's refund policy: items can be returned within 45 days with a receipt.",
    "Acme Corp support hours are 9am-5pm ICT, Monday to Friday.",
    "Acme Corp's Pro plan costs $29 per month and includes 5 seats.",
    "Acme Corp headquarters are located in Phnom Penh, Cambodia.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def retrieve(query, k=1):
    scores = cosine_similarity(vec.transform([query]), doc_vecs).ravel()
    return [docs[i] for i in scores.argsort()[::-1][:k]]

def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(f"Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:")

q = "How many days do I have to return an item at Acme Corp?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q, k=2))

No context: I do not have access to specific return policies for Acme Corp. To find out how many days you have to return an item, you will need to check their official website, return policy page, or contact Acme Corp directly.
With RAG  : 45 days


---
# Hands-on Practice
Complete the `TODO`s. Re-run each cell and compare outputs. Small models are noisy, so try a few variations.

### Exercise 1: Few-shot extraction
Make the model turn a sentence into JSON with keys `name`, `city`, `age`. First try zero-shot, then add **2 examples** to get a clean, consistent format.

In [7]:
text = "Sokha, a 29-year-old engineer, just moved to Siem Reap."

task = "Extract name, city, and age as JSON. Return only valid JSON, without Markdown fences."
zero_shot = f"{task}\n\nText: {text}\nJSON:"
print("Zero-shot:", generate(zero_shot, max_tokens=60))

# Two examples added before the final input (few-shot)
prompt = f"""{task}

Text: Dara is 34 and lives in Battambang.
JSON: {{"name": "Dara", "city": "Battambang", "age": 34}}

Text: Linh, a 22-year-old student, recently relocated to Phnom Penh.
JSON: {{"name": "Linh", "city": "Phnom Penh", "age": 22}}

Text: {text}
JSON:"""

print("Few-shot:", generate(prompt, max_tokens=60))

Zero-shot: {"name": "Sokha", "city": "Siem Reap", "age": 29}
Few-shot: {"name": "Sokha", "city": "Siem Reap", "age": 29}


### Exercise 2: Chain-of-Thought
The puzzle below often trips up direct answers. Compare a **direct** prompt with a **CoT** prompt. Which is right? (Answer: 0.05, i.e. 5 cents)

In [8]:
puzzle = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much is the ball?"

direct = f"Q: {puzzle}\nGive only the final answer in dollars, with no explanation.\nA:"
cot    = f"Q: {puzzle}\nExplain in at most 3 short steps, then give the final answer in dollars.\nA:"

print("Direct:", generate(direct, max_tokens=40))
print("CoT   :", generate(cot, max_tokens=600))

Direct: 0.05
CoT   : 1. Define variables: Let $B$ be the cost of the bat and $L$ be the cost of the ball.
2. Set up equations: $B + L = 1.10$ and $B = L + 1.00$.
3. Solve for $L$: Substitute the second equation into the first: $(L + 1.00) + L = 1.10$, which simplifies to $2L + 1.00 = 1.10$.

$2L = 0.10$
$L = 0.05$

$0.05


### Exercise 3: Self-Consistency
Write a function `self_consistent(prompt, k)` that samples `k` answers at `temperature=0.8` and returns the majority answer. Test it on the puzzle above with `k=1` vs `k=7`. Does voting help?

*Hint:* ask the model to end with `Answer: <number>` so extracting the answer is easy.

In [9]:
def self_consistent(prompt, k=5):
    # sample k outputs at temperature 0.8, extract each final answer, return the most common one
    answers = []
    for _ in range(k):
        out = generate(prompt, max_tokens=600, temperature=0.8)
        m = re.search(
            r"^\s*(?:\*\*)?Answer:(?:\*\*)?\s*(?:\*\*)?\$?\s*"
            r"(-?(?:\d+(?:\.\d+)?|\.\d+))(?:\*\*)?\s*\.?\s*$",
            out.strip(), re.IGNORECASE | re.MULTILINE,
        )
        ans = m.group(1) if m else None
        if ans is not None:
            answers.append(str(round(float(ans), 4)))   # normalize "0.05" / "0.050" / ".05"
    if not answers:
        raise ValueError("No final answers could be parsed. Inspect the model output and rerun.")
    print("Votes:", Counter(answers))
    return Counter(answers).most_common(1)[0][0]

cot_prompt = (f"Q: {puzzle}\nReason in at most 3 short steps, then end on a separate line with 'Answer: <number>' "
              f"where the number is in dollars. Use plain text, no Markdown, and no text after that line.\nA:")

print(self_consistent(cot_prompt, k=1))
print(self_consistent(cot_prompt, k=7))

Votes: Counter({'0.05': 1})
0.05
Votes: Counter({'0.05': 7})
0.05


### Exercise 4: RAG
1. Add **two new facts** of your own to `docs` (re-fit the vectorizer after).
2. Ask a question that only your facts can answer.
3. Compare the answer with and without retrieval. What happens when the answer is **not** in the docs? Edit the RAG prompt so the model says *"I don't know"* in that case.

In [10]:
# 1. Extend docs with two new facts and rebuild the vectorizer / document vectors
docs = docs[:4] + [
    "Acme Corp's mascot is a purple elephant named Tuktuk.",
    "Acme Corp was founded in 2012 by Chantha Vong.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

# 2. A question that only our facts can answer
q = "What is the name of Acme Corp's mascot?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

# 3. What if the answer is NOT in the docs? The original prompt may make the model guess.
q_missing = "Who is the CEO of Acme Corp?"
print("\nNot in docs (original prompt):", rag_answer(q_missing))

# Improved RAG prompt: tell the model to say "I don't know" when the context lacks the answer
def rag_answer_safe(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(
        "Answer the question using only the context below. "
        "If the context does not contain the answer, reply exactly: I don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )

print("Not in docs (improved prompt) :", rag_answer_safe(q_missing))
print("In docs (improved prompt)     :", rag_answer_safe(q))

No context: Please provide the text or context you are referring to so I can tell you the name of Acme Corp's mascot.
With RAG  : Tuktuk

Not in docs (original prompt): The context does not mention who the CEO of Acme Corp is.
Not in docs (improved prompt) : I don't know.
In docs (improved prompt)     : Tuktuk


Observations
- Exercise 1: Both zero-shot and few-shot produced correct JSON.
- Exercise 2: Direct and CoT prompting both correctly answered $0.05.
- Exercise 3: Voting did not improve accuracy in this run because the single sample was already correct. All seven samples agreed on 0.05.
- Exercise 4: RAG supplied the mascot’s name, Tuktuk. The improved prompt returned “I don't know” when the CEO information was absent from the context.

### Challenge (optional)
Pick a task from your own work (e.g. classify support tickets, summarize, translate to Khmer). Design a prompt, then improve it by adding **a role, a format constraint, and examples**. Note which change helped most.

---
## Recap
- **Few-shot**: examples fix format and style
- **CoT**: reasoning steps help multi-step problems
- **Self-consistency**: voting across samples improves reliability, at k× the cost
- **RAG**: grounds answers in your own data

Next: when prompting is not enough, **fine-tuning** updates the weights.